> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# 09_2 — モジュール全層のMP両端保持 × DE
**未実行**。UP・DOWN・GATE・QKVOごとに全ブロックの対象行列を同時LRAし、グループをまたいで累積しません。

主条件：Hのみ／L∪H × DEなし／あり。追加対照：各行列のL∪Hと同rankの上位保持。
2モデル × 4グループ × 2(DE) × 3保持方法 = 48評価＋各モデルbaseline。DE間ではrankは揃えません。
同rank対照は各DE条件内で行列ごとに揃います。個別行列のrank・端点・除外率も保存します。

DE前BEMA・DE後分散1、端点補正とGemma読み込みは09を継承。正方/near-hard-edgeではLなし。
09の単独行列実験1/2/3は09に残し、このノートブックは全層同時処理に専念します。

## 実行
VS CodeでColab GPUを選択し、初回の依存関係導入後、設定のRUN_EXPERIMENTS=Trueで実行。
L4でも開始できますが全層SVDとPPL評価は重い処理です。精度・評価文章は短縮しません。
APIキーは非表示入力。既存08の固定revisionと評価tokensを使用。

各グループの6近似をローカルディスクに一時保存してSVDを共有します。空き容量を事前検査。
結果はDriveへ逐次保存し再開可能。完了グループの一時近似は削除します。
ランタイムが変わるとキャッシュは消えますが、完了済みPPL評価は再利用します。


In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass
INSTALL_DEPENDENCIES = False
CLONE_IF_MISSING = True
REPO = Path('/content/RMT_utils') if Path('/content').exists() else Path.home() / 'GitHub/RMT_utils'
if not (REPO / 'funcs1.py').exists():
    if not CLONE_IF_MISSING:
        raise FileNotFoundError(f'RMT_utils not found: {REPO}')
    subprocess.run(['git', 'clone', 'https://github.com/ryy1210/RMT_utils', str(REPO)], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-notebooks.txt')], check=True)
sys.path.insert(0, str(REPO))
# 必要なときだけ入力。キーをセル出力・設定ファイルに保存しない。
if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN (cached loginなら空欄): ')
    if token: os.environ['HF_TOKEN'] = token
print('Repository:', REPO)

In [ ]:
import json, hashlib, math, gc, time, re, platform, importlib.metadata, inspect, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import funcs1
from funcs1 import bema_algorithm1_from_eigenvalues, bounds, _dyson_from_svd
MOUNT_DRIVE = True
if MOUNT_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').exists():
    DATA_ROOT=Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    base=Path.home()/'Library/CloudStorage/GoogleDrive-ryoya.zushi1210@gmail.com'
    candidates=[p/'TUS/hashiguchi/data' for p in base.iterdir() if unicodedata.normalize('NFC',p.name)=='マイドライブ']
    if len(candidates)!=1: raise FileNotFoundError('DATA_ROOTをDriveのdataフォルダに設定してください')
    DATA_ROOT=candidates[0]
LEGACY_OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'two-edge-module-all-v1'
OUTPUT=DATA_ROOT/'ibis2026_de_lra'/'two-edge-module-all-v1'
MODELS={'llama':'meta-llama/Llama-3.2-3B','gemma':'google/gemma-3-4b-pt'}
MODULES=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
GROUPS={'UP':['up_proj'],'DOWN':['down_proj'],'GATE':['gate_proj'],'QKVO':['q_proj','k_proj','v_proj','o_proj']}
RETENTION=['H','LH','top_same_rank_LH']
CACHE_ROOT=Path('/content/rmt09_2_cache') if Path('/content').exists() else Path('/tmp/rmt09_2_cache')
DELETE_CACHE_AFTER_GROUP=True
RUN_EXPERIMENTS=False
ALL_REQUESTED_COMPLETED=False
BEMA_ALPHA=0.2; TW_BETA=0.1
LOWER_FRACTIONS=[0.05,0.10,0.20]
RANDOM_REPEATS=3; SEED=42
SEQ_LEN=1024; BATCH_SIZE=1
SVD_DEVICE="cuda"  # GPU thin SVD。既存DEの推定式に分解結果を渡して重複SVDを避ける
AUTO_DISCONNECT=False
PUBLISH_WANDB=False
WANDB_ENTITY='ryoya-zushi1210-tokyo-university-of-science'
WANDB_PROJECT='IBIS2026-DE-LRA'

## 固定評価入力と保存先

In [ ]:
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));tmp.replace(path)
def safe_exp(x): return math.exp(x) if x<709 else float('inf')
def atomic_csv(path,df):
    tmp=path.with_suffix('.tmp.csv');df.to_csv(tmp,index=False);tmp.replace(path)
manifest_path=LEGACY_OUTPUT/'manifest.json'
if not manifest_path.exists(): raise FileNotFoundError(f'08のmanifestがありません: {manifest_path}')
legacy=json.loads(manifest_path.read_text())['settings']
REVISIONS=legacy['revisions'];TOTAL=legacy['total']
assert all(re.fullmatch('[0-9a-f]{40}',REVISIONS[k]) for k in MODELS)
TOKENS={};TOKEN_HASH={}
for key in MODELS:
    assert legacy['models'][key]==MODELS[key]
    path=LEGACY_OUTPUT/f'{key}_tokens.npy'
    receipt=json.loads((LEGACY_OUTPUT/f'{key}_tokens.json').read_text())
    assert receipt['sha256']==sha(path), '08 token checksum mismatch'
    TOKENS[key]=np.load(path)
    assert TOKENS[key].ndim==2 and TOKENS[key].shape[1]==SEQ_LEN
    TOKEN_HASH[key]=sha(path)
OUTPUT.mkdir(parents=True,exist_ok=True)
print('固定token blocks:',{k:len(v) for k,v in TOKENS.items()})

## 09と共通のDE・SVD・端点判定

In [ ]:
def decompose(weight):
    transposed=weight.shape[0]>weight.shape[1]
    w=(weight.T if transposed else weight).float().to(SVD_DEVICE)
    u,s,v=torch.linalg.svd(w,full_matrices=False)
    # 既存DEと同じ推定式。GPU thin SVD結果をfloat64で渡す。
    y,x,z=_dyson_from_svd(w.cpu().double().numpy(),u.cpu().double().numpy(),s.cpu().double().numpy(),v.cpu().double().numpy())
    ud,sd,vd=torch.linalg.svd(torch.as_tensor(y,device=SVD_DEVICE,dtype=torch.float32),full_matrices=False)
    return {False:(u,s,v,None,None,transposed),True:(ud,sd,vd,x,z,transposed)}
def spectral_info(parts,use_de):
    u,s,v,x,z,t=parts;p=len(s);n=v.shape[1]
    eig=s.cpu().double().numpy()**2/n
    fit=bema_algorithm1_from_eigenvalues(eig,p,n,alpha=BEMA_ALPHA,beta=TW_BETA)
    variance=1.0 if use_de else float(fit['sigma2_hat'])
    upper=float(fit['threshold']/fit['sigma2_hat']*variance)
    a,b,sm,sp,c,status,intervals=bounds(p,n,variance,upper)
    _,lower,upper=next(q for q in intervals if q[0]=='matched_upper')
    L=np.flatnonzero(eig<lower);H=np.flatnonzero(eig>upper)
    B=np.flatnonzero((eig>=lower)&(eig<=upper))
    assert len(L)+len(H)+len(B)==p
    return dict(p=p,n=n,sigma2=variance,lower=lower,upper=upper,lower_status=status,L=L,B=B,H=H,
                lower_fraction=len(L)/p,upper_fraction=len(H)/p)
def reconstruct(parts,indices):
    u,s,v,x,z,t=parts
    idx=torch.as_tensor(indices,device=s.device,dtype=torch.long)
    result=(u[:,idx]*s[idx])@v[idx,:]
    if x is not None:
        result*=torch.as_tensor(np.sqrt(x),device=result.device,dtype=result.dtype)[:,None]
        result*=torch.as_tensor(np.sqrt(z),device=result.device,dtype=result.dtype)[None,:]
    if t: result=result.T
    if not torch.isfinite(result).all(): raise FloatingPointError('Nonfinite reconstruction')
    return result


In [ ]:
GEMMA_LOADER_VERSION='conditional_verified_v2'
def result_root(key): return OUTPUT/key
def check_loading_info(info):
    if not isinstance(info,dict): raise RuntimeError('Loading report missing')
    errors={k:info.get(k) for k in ['missing_keys','unexpected_keys','mismatched_keys','error_msgs'] if info.get(k)}
    if errors: raise RuntimeError(f'Pretrained checkpoint did not load completely: {errors}')

def extract_gemma_text(full):
    from transformers import Gemma3ForCausalLM
    # 正式クラスでロードした実Tensorを共有。meta上の空モデルへ全モジュールを置換する。
    decoder=full.model.language_model
    with torch.device('meta'):
        text=Gemma3ForCausalLM(full.config.text_config)
    text.model=decoder
    text.lm_head=full.lm_head
    text.generation_config=full.generation_config
    if any(p.is_meta for p in text.parameters()) or any(b.is_meta for b in text.buffers()):
        raise RuntimeError('Unloaded meta tensors remain')
    return text

def model_load(key):
    kwargs=dict(revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'),torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True,output_loading_info=True)
    if key=='gemma':
        from transformers import Gemma3ForConditionalGeneration
        # 修正: checkpointのarchitectureと一致するクラスを使う。text_configだけを渡さない。
        full,info=Gemma3ForConditionalGeneration.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
        model=extract_gemma_text(full)
        del full; gc.collect()
    else:
        model,info=AutoModelForCausalLM.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
    actual=sum(p.numel() for p in model.parameters())
    if actual!=TOTAL[key]: raise ValueError(f'Parameter count mismatch {actual} != {TOTAL[key]}')
    if any(p.is_meta for p in model.parameters()): raise RuntimeError('Unloaded model parameters')
    root=result_root(key); root.mkdir(parents=True,exist_ok=True)
    write_json(root/'loading_report.json',dict(model=MODELS[key],revision=REVISIONS[key],
        loader_version=GEMMA_LOADER_VERSION if key=='gemma' else 'auto_verified_v2',
        transformers=importlib.metadata.version('transformers'),parameter_count=actual,
        missing_keys=[],unexpected_keys=[],mismatched_keys=[],config_hash=CONFIG_HASH))
    model.eval(); model.config.use_cache=False
    return model.to('cuda')

def evaluate(model,tokens):
    losses=[]; counts=[]
    for start in range(0,len(tokens),BATCH_SIZE):
        ids=torch.as_tensor(tokens[start:start+BATCH_SIZE],device=next(model.parameters()).device)
        logits=model(input_ids=ids,use_cache=False).logits
        sums=torch.zeros(ids.shape[0],device=ids.device,dtype=torch.float64)
        for pos in range(0,ids.shape[1]-1,64):
            end=min(pos+64,ids.shape[1]-1)
            loss=F.cross_entropy(logits[:,pos:end].float().transpose(1,2),ids[:,pos+1:end+1],reduction='none')
            if not torch.isfinite(loss).all(): raise FloatingPointError('Nonfinite NLL')
            sums+=loss.double().sum(1)
        losses.extend(sums.cpu().tolist()); counts.extend([ids.shape[1]-1]*ids.shape[0]); del logits
    return np.asarray(losses),np.asarray(counts)

def resolve(model,name):
    found=[m for k,m in model.named_modules() if isinstance(m,torch.nn.Linear) and (k==name or k.endswith('.'+name))]
    if len(found)!=1: raise ValueError(f'Ambiguous module {name}')
    return found[0]

In [ ]:
evaluate=torch.inference_mode()(evaluate)
def function_fingerprint(f):
    # ipykernelの一時filename・セル番号は再起動で変わるのでhashに含めない。
    import types
    def normalize(code):
        return [code.co_code.hex(),code.co_names,code.co_varnames,
                [normalize(x) if isinstance(x,types.CodeType) else repr(x) for x in code.co_consts]]
    return digest(normalize(f.__code__))

def read_result(path):
    if not (path/'summary.json').exists(): return None
    obj=json.loads((path/'summary.json').read_text())
    assert obj['config_hash']==CONFIG_HASH
    f=path/'losses.npz'
    assert sha(f)==obj['loss_hash'], 'Result checksum mismatch'
    with np.load(f) as z: loss=z['loss'];count=z['count']
    assert len(loss)==len(TOKENS[obj['model']]) and np.all(count==SEQ_LEN-1) and np.isfinite(loss).all()
    return obj

def save_result(path,key,loss,count,extra):
    path.mkdir(parents=True,exist_ok=True)
    with open(path/'losses.tmp','wb') as f: np.savez_compressed(f,loss=loss,count=count)
    (path/'losses.tmp').replace(path/'losses.npz')
    value=safe_exp(float(loss.sum()/count.sum()))
    obj=dict(model=key,config_hash=CONFIG_HASH,ppl=value if np.isfinite(value) else None,
             ppl_overflow=not np.isfinite(value),loss_hash=sha(path/'losses.npz'),**extra)
    write_json(path/'summary.json',obj)
    return obj

In [ ]:

def initialize_manifest():
    global CONFIG_HASH
    settings=dict(version='09_2-v1',models=MODELS,revisions=REVISIONS,total=TOTAL,token_hash=TOKEN_HASH,
        groups=GROUPS,retention=RETENTION,seq_len=SEQ_LEN,batch=BATCH_SIZE,dtype='bfloat16',
        svd_device=SVD_DEVICE,bema_alpha=BEMA_ALPHA,tw_beta=TW_BETA,funcs1_hash=sha(Path(funcs1.__file__)),
        implementation={f.__name__:function_fingerprint(f) for f in [decompose,spectral_info,reconstruct,
            model_load,extract_gemma_text,evaluate.__wrapped__,group_masks,prepare_cache,run,save_result,read_result]})
    CONFIG_HASH=digest(settings)
    path=OUTPUT/'manifest.json'
    if path.exists() and json.loads(path.read_text())['config_hash']!=CONFIG_HASH:
        raise ValueError('設定/実装が保存済み結果と異なります。OUTPUTのTAGを変更してください')
    write_json(path,dict(config_hash=CONFIG_HASH,settings=settings))

def group_masks(info):
    both=np.union1d(info['L'],info['H'])
    return {'H':info['H'],'LH':both,'top_same_rank_LH':np.arange(len(both))}

def cache_read(path):
    receipt=path.with_suffix('.json')
    if not path.exists() or not receipt.exists():return None
    obj=json.loads(receipt.read_text())
    if obj['config_hash']!=CONFIG_HASH or obj['sha256']!=sha(path):raise ValueError('近似キャッシュ不整合')
    return obj

def prepare_cache(key,group,matrices):
    import shutil
    root=CACHE_ROOT/CONFIG_HASH/key/group;root.mkdir(parents=True,exist_ok=True)
    # BF16の6条件分＋生成中一時ファイル。キャッシュ済みも含める保守的な容量判定。
    required=sum(m.weight.numel()*2 for _,m in matrices)*len(RETENTION)*2
    existing=sum(p.stat().st_size for p in root.glob('*.pt'))
    if shutil.disk_usage(root).free < max(0,required-existing)+512*1024**2:
        raise RuntimeError(f'近似キャッシュ用ディスク不足: 最大約{required/1e9:.1f}GB必要。CACHE_ROOTを変更してください')
    records=[]
    for index,(name,module) in enumerate(matrices):
        paths={(de,label):root/f'{digest([name,de,label])[:24]}.pt' for de in [False,True] for label in RETENTION}
        receipts={k:cache_read(p) for k,p in paths.items()}
        if all(receipts.values()):
            records.extend(v['record'] for v in receipts.values());continue
        print(f'  SVD [{index+1}/{len(matrices)}] {name}',flush=True)
        original=module.weight.detach().cpu().clone();parts=decompose(original)
        for de in [False,True]:
            info=spectral_info(parts[de],de);masks=group_masks(info)
            for label in RETENTION:
                path=paths[de,label]
                if receipts[de,label]:records.append(receipts[de,label]['record']);continue
                idx=masks[label];r=len(idx);p,n=info['p'],info['n']
                value=original.float() if r==p else reconstruct(parts[de],idx).cpu()
                error=float(torch.linalg.vector_norm(value-original.float())/torch.linalg.vector_norm(original.float()))
                value=value.to(original.dtype)
                if not torch.isfinite(value).all():raise FloatingPointError('BF16変換後の近似に非有限値')
                tmp=path.with_suffix('.tmp');torch.save(value,tmp);tmp.replace(path)
                rec=dict(name=name,de=de,label=label,rank=r,p=p,n=n,lower=info['lower'],upper=info['upper'],sigma2=info['sigma2'],
                    lower_status=info['lower_status'],lower_fraction=info['lower_fraction'],upper_fraction=info['upper_fraction'],
                    n_lower=len(info['L']),n_upper=len(info['H']),relative_frobenius_error=error,
                    dense_parameters=p*n,factor_parameters=r*(p+n),saved_parameters=max(0,p*n-r*(p+n)),
                    indices=idx.tolist(),cache_file=str(path))
                write_json(path.with_suffix('.json'),dict(config_hash=CONFIG_HASH,sha256=sha(path),record=rec));records.append(rec)
                del value
        del parts,original;gc.collect();torch.cuda.empty_cache()
    return root,records

def run():
    global ALL_REQUESTED_COMPLETED
    ALL_REQUESTED_COMPLETED=False
    initialize_manifest()
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():raise RuntimeError('BF16対応CUDA GPUを選択してください')
    write_json(OUTPUT/'environment.json',dict(gpu=torch.cuda.get_device_name(),python=platform.python_version(),
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','datasets']}))
    import shutil
    for key in MODELS:
        jobs=[(group,de,label) for group in GROUPS for de in [False,True] for label in RETENTION]
        def jobpath(group,de,label):return OUTPUT/key/'groups'/group/f'{int(de)}_{label}'
        pending=[j for j in jobs if read_result(jobpath(*j)) is None]
        if not pending and read_result(OUTPUT/key/'baseline'):
            print(key,'全条件完了。モデル読み込み不要');continue
        model=model_load(key)
        try:
            baseline=read_result(OUTPUT/key/'baseline')
            if baseline is None:
                loss,count=evaluate(model,TOKENS[key]);baseline=save_result(OUTPUT/key/'baseline',key,loss,count,dict(kind='baseline'))
            if baseline['ppl'] is None or baseline['ppl']>1000:raise RuntimeError('Baseline異常。圧縮前に停止')
            all_linear={n:m for n,m in model.named_modules() if isinstance(m,torch.nn.Linear)}
            for group,targets in GROUPS.items():
                todo=[j for j in pending if j[0]==group]
                if not todo:continue
                matrices=[(n,m) for n,m in all_linear.items() if n.split('.')[-1] in targets]
                if not matrices:raise ValueError(f'{group}: 対象なし')
                # 全blockの各対象モジュールが1個ずつあることを確認。
                expected=getattr(model.config,'num_hidden_layers',None)
                for target in targets:
                    selected=[n for n,m in matrices if n.split('.')[-1]==target]
                    blocks=[int(re.search(r'layers\.(\d+)\.',n).group(1)) for n in selected]
                    if expected is None or sorted(blocks)!=list(range(expected)):
                        raise ValueError(f'{target}: 全層の網羅を確認できません: {blocks}')
                print(f'{key} {group}: {len(matrices)}行列を同時LRA、残り{len(todo)}条件',flush=True)
                cache,records=prepare_cache(key,group,matrices)
                groupdir=OUTPUT/key/'groups'/group;groupdir.mkdir(parents=True,exist_ok=True)
                write_json(groupdir/'matrix_plan.json',dict(config_hash=CONFIG_HASH,records=records))
                # 全グループ分を保持せず、現在のグループの元重みのみCPUに退避。
                originals={n:m.weight.detach().cpu().clone() for n,m in matrices}
                try:
                    for _,de,label in todo:
                        chosen=[r for r in records if r['de']==de and r['label']==label]
                        if {r['name'] for r in chosen}!=set(originals):raise ValueError('近似の網羅性不一致')
                        start=time.time()
                        try:
                            for rec in chosen:
                                path=Path(rec['cache_file']);cache_read(path)
                                value=torch.load(path,map_location='cpu',weights_only=True)
                                module=all_linear[rec['name']]
                                if value.shape!=module.weight.shape or not torch.isfinite(value).all():raise ValueError('近似shape/値が異常')
                                with torch.no_grad():module.weight.copy_(value.to(module.weight.device))
                                del value
                            loss,count=evaluate(model,TOKENS[key])
                            saved=sum(r['saved_parameters'] for r in chosen);dense=sum(r['dense_parameters'] for r in chosen)
                            dest=jobpath(group,de,label)
                            dest.mkdir(parents=True,exist_ok=True)
                            atomic_csv(dest/'matrices.csv',pd.DataFrame([{k:v for k,v in r.items() if k not in ['indices','cache_file']} for r in chosen]))
                            save_result(dest,key,loss,count,dict(group=group,de=de,label=label,matrix_count=len(chosen),
                                sum_rank=sum(r['rank'] for r in chosen),params_saved=saved,target_parameters=dense,
                                reduction_target=saved/dense,reduction_model=saved/TOTAL[key],
                                baseline_ppl=baseline['ppl'],seconds=time.time()-start))
                            print(f'  saved DE={de} {label}',flush=True)
                        finally:
                            # 例外・中断時も変更した全行列を復元。次条件に累積させない。
                            with torch.no_grad():
                                for n,m in matrices:m.weight.copy_(originals[n].to(m.weight.device))
                    if DELETE_CACHE_AFTER_GROUP:shutil.rmtree(cache)
                finally:
                    del originals;gc.collect();torch.cuda.empty_cache()
        finally:
            del model;gc.collect();torch.cuda.empty_cache()
    ALL_REQUESTED_COMPLETED=True


## 軽量検証・本番実行

In [ ]:

def smoke_test():
    global SVD_DEVICE
    old=SVD_DEVICE;SVD_DEVICE='cpu'
    try:
        torch.manual_seed(17)
        for shape in [(24,80),(80,24),(24,24)]:
            w=torch.randn(*shape);parts=decompose(w)
            for de in [False,True]:
                assert torch.allclose(reconstruct(parts[de],np.arange(min(shape))),w,atol=3e-5,rtol=3e-5)
                info=spectral_info(parts[de],de);masks=group_masks(info)
                assert len(masks['LH'])==len(masks['top_same_rank_LH'])
                assert not np.intersect1d(masks['LH'],info['B']).size
        print('PASS: 全成分復元・両端mask・同rank対照')
    finally:SVD_DEVICE=old
smoke_test()
if RUN_EXPERIMENTS:run()
else:print('未実行。RUN_EXPERIMENTS=Trueで開始。主比較32条件＋対照16条件。')


## 結果の集計
PPLは低いほど良い。DE改善率=100×(1−PPL_DE/PPL_raw)、正がDE優位。
DE間の削減率は異なるため必ず併記。同rank上位保持との比較は各DE条件内。
未完了は未完了として表示。削減率は理論上の因子化パラメータ数に基づき、実行時はdense行列で推論します。


In [ ]:

initialize_manifest()
rows=[]
for key in MODELS:
    for group in GROUPS:
        for de in [False,True]:
            for label in RETENTION:
                obj=read_result(OUTPUT/key/'groups'/group/f'{int(de)}_{label}')
                rows.append(({**obj,'status':'complete'} if obj else None) or dict(model=key,group=group,de=de,label=label,ppl=None,status='pending'))
RESULTS=pd.DataFrame(rows)
RESULTS['status']=RESULTS.status.fillna('complete')
atomic_csv(OUTPUT/'results.csv',RESULTS)
display(RESULTS[[c for c in ['model','group','de','label','status','ppl','reduction_target','reduction_model','sum_rank'] if c in RESULTS]])
done=RESULTS[RESULTS.status=='complete'].copy()
if not done.empty:
    if done.ppl.isna().any():print('PPLの数値範囲を超えた条件あり。図には表示されません。')
    pairs=[]
    for (key,group,label),g in done.groupby(['model','group','label']):
        if set(g.de)!={False,True}:continue
        a=g[g.de==False].iloc[0];b=g[g.de==True].iloc[0]
        pairs.append(dict(model=key,group=group,label=label,ppl_raw=a.ppl,ppl_DE=b.ppl,
            DE_gain_percent=100*(1-b.ppl/a.ppl),reduction_raw=a.reduction_target,reduction_DE=b.reduction_target))
    PAIRS=pd.DataFrame(pairs);atomic_csv(OUTPUT/'DE_comparisons.csv',PAIRS);display(PAIRS)
    within=done.pivot(index=['model','group','de'],columns='label',values='ppl')
    if {'H','LH','top_same_rank_LH'}<=set(within):
        within['lower_addition_PPL_gain']=within.H-within.LH
        within['same_rank_PPL_gain']=within.top_same_rank_LH-within.LH
        atomic_csv(OUTPUT/'retention_comparisons.csv',within.reset_index());display(within)
    for key,g in done.groupby('model'):
        fig,axes=plt.subplots(1,2,figsize=(12,4))
        for (de,label),part in g.groupby(['de','label']):
            axes[0].scatter(part.reduction_target*100,part.ppl,label=f'{label}, DE={de}',marker='s' if de else 'o')
            for _,r in part.iterrows():axes[0].annotate(r.group,(r.reduction_target*100,r.ppl),fontsize=7)
        axes[0].set(xlabel='Target-group reduction (%)',ylabel='PPL (log scale)',yscale='log',title=key)
        axes[0].legend(fontsize=7)
        for (de,label),part in g.groupby(['de','label']):
            values=part.set_index('group').reindex(GROUPS)
            axes[1].plot(list(GROUPS),values.ppl,marker='o' if not de else 's',label=f'{label}, DE={de}')
        axes[1].set(ylabel='PPL (log scale)',yscale='log',title='All blocks replaced within each group')
        axes[1].legend(fontsize=7);fig.tight_layout();fig.savefig(OUTPUT/f'{key}_module_ppl.png',dpi=160);plt.show()


In [ ]:
if PUBLISH_WANDB and not RESULTS.empty:
    import wandb
    if not os.environ.get('WANDB_API_KEY'):
        os.environ['WANDB_API_KEY']=getpass.getpass('WANDB_API_KEY: ')
    with wandb.init(entity=WANDB_ENTITY,project=WANDB_PROJECT,job_type='two-edge-lra',config={'config_hash':CONFIG_HASH}) as wb:
        wb.log({'results':wandb.Table(dataframe=RESULTS.fillna(''))})
if AUTO_DISCONNECT and RUN_EXPERIMENTS and ALL_REQUESTED_COMPLETED:
    # 計算セルが例外で止まった場合はRun Allもここに到達しません。
    from google.colab import runtime
    runtime.unassign()